# Feature Encoders: In-Domain vs. Out-of-Domain Performance

Experiment E2 (`docs/experiments/cross_domain_experiments_details.md`): the same graph model (Tissue-Region-Net,
TRN-Ref configuration) trained on node embeddings from two different patch encoders —
**Prov-GigaPath** (pathology foundation model) and **ResNet-50** (ImageNet-pretrained). Everything else
(splits, seed, checkpoint and threshold selection) is identical. The question is whether the encoder changes
in-domain performance, out-of-domain robustness, or both.

Note: the two encoders use separately built graph corpora (`context_aware_gsdb_inflammation` vs.
`context_aware_runtime_non_hist_gsdb_inflammation`, see the data table below); differences between encoders
therefore include everything that differs between those corpora.

**Protocol** (see `docs/experiments/cross_domain_experiments_details.md`)

| | Split | Scanner | Stains | Use |
|---|---|---|---|---|
| Training | `train_HE` | 1 | H&E | model fitting |
| Validation | `val_HE` | 1 | H&E | checkpoint selection and decision threshold |
| ID test | `test` | 1 | H&E, MG, PAS | in-domain test |
| OOD test | `test_scanner2` | 2 | H&E, MG, PAS | out-of-domain test (prediction only) |

The decision threshold is fixed on the validation split; no recalibration is applied to either test set.
The OOD set differs from the training domain in **scanner** and in **stain composition** (ID test 81 % H&E,
OOD balanced across H&E, MG, PAS); the stain-stratified section separates the two.

ΔAUC = AUC(OOD) − AUC(ID) = −Gap, where the generalization gap used in the docs and the thesis is
Gap = AUC(S1) − AUC(S2); a ΔAUC of −5.6 pp therefore corresponds to Gap = 0.056.

**Statistics.** Slides are nested in patients (in the OOD set every patient contributes one slide per stain),
so all confidence intervals are 95 % percentile intervals from a **patient-level cluster bootstrap**
(B = `N_BOOT`). ΔAUC between domains uses independent bootstraps of the two patient-disjoint test sets;
comparisons between models within a domain use a paired bootstrap on the slides both models predicted, with
two-sided bootstrap *p*-values, Holm-corrected across all comparisons. Threshold-dependent metrics use the
validation threshold (positive if *p* > *t*), as in `_compute_slide_binary_metrics`. Validation AUCs come from
the evaluation run (`validation_thresholds.json`, 39 usable slides) and have no CI.

**Figures**

0. Training curves of both encoders with the selected checkpoint
1. Scorecard: all slide-level metrics per encoder for ID, OOD and the change (OOD − ID)
2. Classification outcomes (TP / FN, TN / FP) and confusion matrices
3. AUC along validation → ID → OOD, ΔAUC (OOD − ID), and the encoder effect on robustness (ΔΔAUC)
4. ID vs. OOD AUC, ROC curves, stain-stratified AUC, threshold transfer, slide-score distributions

Missing evaluations are shown as gaps; the data section prints the `evaluate_model.py` commands that fill them.

CAMELYON17 (E2 part 2: validation = center 3, test = center 4, both unseen centers) is not included yet: no
evaluation outputs exist for those runs so far.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
import yaml
from matplotlib.ticker import MaxNLocator
from sklearn.metrics import roc_auc_score, roc_curve

REPO_ROOT = Path("/home/stud/hweber/master_code")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.plot_style import IEEE_COLORS, apply_ieee_style, figsize

SPLITS_DIR = REPO_ROOT / "data/splits/gsdb/seed_42"
OUT_DIR = REPO_ROOT / "notebooks/results/encoders"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Configuration

In [ ]:
# {encoder: {"ID": evaluation path, "OOD": evaluation path, "history": training history, "meta": training meta}}
# Paths are relative to the repository root or absolute. An evaluation path may point to the evaluation
# folder (containing split_<split>/) or directly to the split_<split> folder. None = not evaluated.
# ID and OOD must come from the same checkpoint -- the notebook warns if their validation thresholds differ.
# "meta" is used to show the graph corpus and to print the command for missing evaluations.
RUNS = {
    "Prov-GigaPath": {
        "ID":      "results/evaluations/e2-gsdb-bin-gigapath_inflammation_dataset-gsdb_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_heonly_20261012_080503",
        "OOD":     "results/test_results/e2-gsdb-bin-gigapath_inflammation_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_trainheonly_None_tissue_region_net_20261017_150610",
        "history": "results/models/training_history_e2-gsdb-bin-gigapath_inflammation_dataset-gsdb_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_gnn_tissue_region_net_graph_pov-gigapath_heonly_segmentation.json",
        "meta":    "results/metas/training_metas/e2-gsdb-bin-gigapath_task-inflammation_heonly_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_dataset-gsdb_model_4a91c1b7_mil_cached_training_method-function_20261012_090310.yaml",
    },
    "ResNet-50": {
        "ID":      "results/evaluations/e2-gsdb-bin-resnet50_inflammation_dataset-gsdb_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_resnet50_gnn_tissue_region_net_graph_resnet50_heonly_20261012_091150",
        "OOD":     "results/test_results/e2-gsdb-bin-resnet50_inflammation_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_resnet50_trainheonly_None_tissue_region_net_20261017_151132",
        "history": "results/models/training_history_e2-gsdb-bin-resnet50_inflammation_dataset-gsdb_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_resnet50_gnn_tissue_region_net_graph_resnet50_heonly_segmentation.json",
        "meta":    "results/metas/training_metas/e2-gsdb-bin-resnet50_task-inflammation_heonly_cfg-model_config_tissue_region_segmentation_region_monitorloss_pseudolabel_resnet50_dataset-gs_6ac047cd_mil_cached_training_method-function_20261012_102535.yaml",
    },
}
DOMAINS = {"ID": "test", "OOD": "test_scanner2"}          # domain label -> split name
DOMAIN_LONG = {"VAL": "Validation (S1 · H&E)", "ID": "ID test (S1)", "OOD": "OOD test (S2)"}
STRATA = [("ID", "HE"), ("OOD", "HE"), ("OOD", "MG"), ("OOD", "PAS")]
REFERENCE = "ResNet-50"            # baseline encoder; pairwise Δ = encoder − ResNet-50
N_BOOT = 2000
SEED = 0
SAVE_PREFIX = "e2"                 # None = do not write figures / tables
MODELS_DIR = REPO_ROOT / "results/models"
META_DIR = REPO_ROOT / "results/metas/training_metas"

## 1 · Data

In [ ]:
def _one(directory, pattern):
    hits = sorted(directory.glob(pattern)) if directory.is_dir() else []
    return hits[0] if hits else None


def resolve_path(path):
    # Repository-relative or absolute path; None stays None. A given path that does not exist is an error.
    if path is None:
        return None
    p = Path(path)
    p = p if p.is_absolute() else REPO_ROOT / p
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p}")
    return p


def resolve_split_dir(path, split):
    # Accepts an evaluation folder (containing split_<split>/) or the split_<split> folder itself.
    p = resolve_path(path)
    if p is None:
        return None
    split_dir = p if p.name == f"split_{split}" else p / f"split_{split}"
    if not _one(split_dir, f"*_{split}_slide_metrics.json"):
        raise FileNotFoundError(f"No *_{split}_slide_metrics.json in {split_dir}")
    return split_dir


def load_split(split_dir, split):
    pred_file = _one(split_dir, f"*_{split}_slide_predictions.csv") or _one(split_dir, f"*_{split}_predictions.csv")
    preds = pd.read_csv(pred_file)[["slide_name", "prob", "label"]]
    if preds["slide_name"].duplicated().any():
        raise ValueError(f"{pred_file} is not slide-level")
    with open(_one(split_dir, f"*_{split}_slide_metrics.json")) as f:
        metrics = json.load(f)
    at_val = metrics.get("at_validation_threshold") or metrics.get("slide_at_validation_threshold")
    val_file = split_dir.parent / "validation_thresholds" / "validation_thresholds.json"
    val_auc = json.load(open(val_file)).get("validation_slide_auroc") if val_file.exists() else None
    return {"preds": preds, "at_val": at_val, "val_auc": val_auc, "source": split_dir}


cohorts = {}
for dom, split in DOMAINS.items():
    c = pd.read_csv(SPLITS_DIR / f"{split}.csv")
    c = c[c["inflammation_status"].isin(["inflamed", "noninflamed"])].reset_index(drop=True)
    c["label"] = (c["inflammation_status"] == "inflamed").astype(int)
    cohorts[dom] = c[["slide_name", "patient_id", "scanner_id", "stain", "label"]]

results, availability = {}, []
for model, paths in RUNS.items():
    for dom, split in DOMAINS.items():
        split_dir = resolve_split_dir(paths.get(dom), split)
        row = {"Model": model, "Domain": dom, "Split": split}
        if split_dir is None:
            availability.append({**row, "Slides": "missing", "Source": "–"})
            continue
        r = load_split(split_dir, split)
        merged = cohorts[dom].merge(r["preds"], on="slide_name", how="left", suffixes=("", "_pred"))
        known = merged["prob"].notna()
        if (merged.loc[known, "label"] != merged.loc[known, "label_pred"]).any():
            raise ValueError(f"Label mismatch between split file and predictions: {model} / {dom}")
        r["prob"] = merged["prob"].to_numpy()
        results[(model, dom)] = r
        availability.append({**row, "Slides": f"{known.sum()}/{len(merged)}",
                             "Source": f"{split_dir.parent.parent.name}/{split_dir.parent.name[:40]}…"})

colors = {m: IEEE_COLORS[i % len(IEEE_COLORS)] for i, m in enumerate(RUNS)}
# Redundant encoding for colour-blind readers: every model also has its own marker and line style.
model_markers = {m: ["o", "s", "^", "D", "v", "P"][i % 6] for i, m in enumerate(RUNS)}
model_lines = {m: ["-", "--", "-.", ":"][i % 4] for i, m in enumerate(RUNS)}
models = [m for m in RUNS if any((m, d) in results for d in DOMAINS)]
val_auc = {m: next((results[(m, d)]["val_auc"] for d in DOMAINS
                    if (m, d) in results and results[(m, d)]["val_auc"] is not None), None) for m in models}
pd.DataFrame(availability).set_index(["Model", "Domain"])

In [ ]:
# Training metas: graph corpus per encoder, and commands for missing evaluations
def find_key(d, key):
    if isinstance(d, dict):
        if key in d:
            return d[key]
        for v in d.values():
            hit = find_key(v, key)
            if hit is not None:
                return hit
    return None


metas = {}
for model, paths in RUNS.items():
    metas[model] = resolve_path(paths.get("meta"))
    corpus = find_key(yaml.safe_load(open(metas[model])), "graph_target_dir") if metas[model] else None
    print(f"{model:15s} corpus: {Path(str(corpus)).name if corpus else 'unknown'}")

todo = [(m, split) for m in RUNS for dom, split in DOMAINS.items() if (m, dom) not in results]
if todo:
    print("\n# Missing evaluations -- run from the repository root:")
    for model in dict.fromkeys(m for m, _ in todo):
        splits = " ".join(s for m, s in todo if m == model)
        meta = metas[model].relative_to(REPO_ROOT) if metas[model] else f"<meta file of {model}>"
        print(f"PYTHONHASHSEED=42 python src/entrypoint/evaluate_model.py --splits {splits} \\\n  --meta_file {meta}")

In [ ]:
# Cohort description (per domain and stain)
rows = []
for dom, c in cohorts.items():
    for stain, g in list(c.groupby("stain")) + [("all", c)]:
        rows.append({"Domain": DOMAIN_LONG[dom], "Stain": stain, "Slides": len(g),
                     "Patients": g["patient_id"].nunique(), "Inflamed": int(g["label"].sum()),
                     "Non-inflamed": int((1 - g["label"]).sum())})
cohort_table = pd.DataFrame(rows).set_index(["Domain", "Stain"])
cohort_table

In [ ]:
# Protocol checks: same validation threshold for ID and OOD, and the confusion matrix recomputed here at that
# threshold equals the one stored by the evaluation pipeline.
def confusion(y, p, t):
    pred = (p > t).astype(int)
    return {"tp": int(((pred == 1) & (y == 1)).sum()), "fn": int(((pred == 0) & (y == 1)).sum()),
            "fp": int(((pred == 1) & (y == 0)).sum()), "tn": int(((pred == 0) & (y == 0)).sum())}


for model in models:
    thresholds = {d: results[(model, d)]["at_val"]["threshold"] for d in DOMAINS if (model, d) in results}
    if len(set(np.round(list(thresholds.values()), 9))) > 1:
        print(f"WARNING {model}: validation thresholds differ between domains {thresholds}")
    for dom in DOMAINS:
        if (model, dom) not in results:
            continue
        r, y = results[(model, dom)], cohorts[dom]["label"].to_numpy()
        ok = ~np.isnan(r["prob"])
        cm = confusion(y[ok], r["prob"][ok], r["at_val"]["threshold"])
        stored = {k: r["at_val"][k] for k in cm}
        status = "ok" if cm == stored else f"MISMATCH stored={stored}"
        print(f"{model:18s} {dom:3s} threshold={r['at_val']['threshold']:.4f}  confusion {cm}  {status}")

## Training curves

Validation curves (scanner-1 H&E) of both encoders. The vertical line marks the saved checkpoint
(max. validation slide F2, ties → last epoch, as in `train.py`), i.e. the weights evaluated on the test sets.

In [ ]:
apply_ieee_style()
W, _ = figsize("double")
histories = {}
for model, paths in RUNS.items():
    history_file = resolve_path(paths.get("history"))
    if history_file is not None:
        histories[model] = json.load(open(history_file))


def selected_epoch(hist):
    meta = hist.get("metadata", {})
    metric = meta.get("monitor_metric")
    if metric:
        values = np.asarray(hist.get(f"val_{metric}") or hist[metric], dtype=float)
        best = values.max() if meta.get("monitor_mode", "max") == "max" else values.min()
        return int(np.flatnonzero(values == best)[-1])
    return int(np.argmin(hist["val_loss"]))


CURVES = [("val_slide_auc", "Slide AUC (%)"), ("val_region_auc", "Region AUC (%)"),
          ("val_slide_f2", "Slide F2 (%)"), ("loss", "Loss")]
fig, axes = plt.subplots(1, len(CURVES), figsize=(W, 2.3))
model_colors = {m: IEEE_COLORS[i % len(IEEE_COLORS)] for i, m in enumerate(RUNS)}
model_marks = {m: ["o", "s", "^", "D"][i % 4] for i, m in enumerate(RUNS)}
for ax, (key, title), letter in zip(axes, CURVES, "abcd"):
    for model, hist in histories.items():
        c = model_colors[model]
        sel = selected_epoch(hist)
        if key == "loss":
            for split, ls, alpha in (("train", "--", 0.55), ("val", "-", 1.0)):
                v = hist[f"{split}_loss"]
                ax.plot(np.arange(1, len(v) + 1), v, color=c, linestyle=ls, alpha=alpha, linewidth=1.2,
                        marker=model_marks[model], markevery=4, markersize=3,
                        markerfacecolor="white" if split == "train" else c)
        elif hist.get(key):
            v = hist[key]
            ax.plot(np.arange(1, len(v) + 1), v, color=c, linewidth=1.3, marker=model_marks[model], markevery=4,
                    markersize=3)
            ax.plot(sel + 1, v[sel], model_marks[model], color=c, markersize=6, markeredgecolor="black",
                    markeredgewidth=0.7, zorder=4)
        ax.axvline(sel + 1, color=c, linestyle="-.", linewidth=0.8, alpha=0.8)
    ax.set_title(f"({letter}) {title}", loc="left", fontsize=8.5)
    ax.set_xlabel("Epoch")
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if key != "loss":
        lo = min(np.min(h[key]) for h in histories.values() if h.get(key))
        ax.set_ylim(max(0, np.floor((lo - 2) / 5) * 5), 101)
fig.legend(handles=[Line2D([], [], color=model_colors[m], marker=model_marks[m], markersize=4, linewidth=1.5,
                           label=m) for m in histories] + [
    Line2D([], [], color="0.3", linestyle="--", alpha=0.55, marker="o", markerfacecolor="white", markersize=3,
           label="training loss (hollow markers)"),
    Line2D([], [], color="0.3", linestyle="-.", marker="o", markerfacecolor="white", markeredgecolor="black",
           markersize=6, label="selected checkpoint (line, black-edged marker)")],
    loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.1, 1, 1))
if SAVE_PREFIX:
    for ext in ("png", "pdf"):
        fig.savefig(OUT_DIR / f"{SAVE_PREFIX}_training.{ext}", bbox_inches="tight")
plt.show()
for model, hist in histories.items():
    sel = selected_epoch(hist)
    print(f"{model:15s} selected epoch {sel + 1} / {len(hist['train_loss'])}")

## 2 · Bootstrap estimates

In [ ]:
rng = np.random.default_rng(SEED)


def cluster_bootstrap(patient_ids, n_boot, rng):
    # Row indices of n_boot patient-level resamples.
    patients, inverse = np.unique(patient_ids, return_inverse=True)
    rows_of = [np.flatnonzero(inverse == k) for k in range(len(patients))]
    return [np.concatenate([rows_of[k] for k in rng.integers(0, len(patients), len(patients))])
            for _ in range(n_boot)]


boot = {dom: cluster_bootstrap(c["patient_id"].to_numpy(), N_BOOT, rng) for dom, c in cohorts.items()}


def auc(y, p):
    ok = ~np.isnan(p)
    y, p = y[ok], p[ok]
    return roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan


def threshold_metrics(y, p, t):
    ok = ~np.isnan(p)
    cm = confusion(y[ok], p[ok], t)
    sens = cm["tp"] / (cm["tp"] + cm["fn"]) if cm["tp"] + cm["fn"] else np.nan
    spec = cm["tn"] / (cm["tn"] + cm["fp"]) if cm["tn"] + cm["fp"] else np.nan
    f1 = 2 * cm["tp"] / (2 * cm["tp"] + cm["fp"] + cm["fn"]) if cm["tp"] + cm["fp"] + cm["fn"] else np.nan
    ppv = cm["tp"] / (cm["tp"] + cm["fp"]) if cm["tp"] + cm["fp"] else np.nan
    npv = cm["tn"] / (cm["tn"] + cm["fn"]) if cm["tn"] + cm["fn"] else np.nan
    return {"AUC": auc(y, p), "Sensitivity": sens, "Specificity": spec, "PPV": ppv, "NPV": npv,
            "Balanced accuracy": np.nanmean([sens, spec]), "F1": f1}


def oracle_balanced_accuracy(y, p):
    # Best balanced accuracy over all thresholds on the evaluated set itself. Diagnostic only: it uses the
    # test labels and is an optimistic upper bound of what any fixed threshold can reach on this set.
    ok = ~np.isnan(p)
    y, p = y[ok], p[ok]
    best = (0.0, np.nan)
    for t in np.concatenate([[-np.inf], np.unique(p)]):
        m = threshold_metrics(y, p, t)
        if m["Balanced accuracy"] > best[0]:
            best = (m["Balanced accuracy"], t)
    return best


def estimate(dom, p, t, subset=None):
    # Point estimate + bootstrap replicates of all metrics on a (stain-)subset of one domain.
    y = cohorts[dom]["label"].to_numpy()
    mask = np.ones(len(y), bool) if subset is None else subset
    point = threshold_metrics(y[mask], p[mask], t)
    reps = [threshold_metrics(y[idx[mask[idx]]], p[idx[mask[idx]]], t) for idx in boot[dom]]
    return point, pd.DataFrame(reps)


def ci(series):
    s = pd.Series(series).dropna()
    return np.percentile(s, [2.5, 97.5]) if len(s) else (np.nan, np.nan)


est = {key: estimate(key[1], r["prob"], r["at_val"]["threshold"]) for key, r in results.items()}


def delta_auc(model):
    # ΔAUC = OOD − ID with independent bootstraps of the two patient-disjoint test sets.
    if not all((model, d) in est for d in ("ID", "OOD")):
        return None
    reps = est[(model, "OOD")][1]["AUC"].to_numpy() - est[(model, "ID")][1]["AUC"].to_numpy()
    return est[(model, "OOD")][0]["AUC"] - est[(model, "ID")][0]["AUC"], ci(reps)


print(f"{N_BOOT} patient-level bootstrap resamples per domain.")

## 3 · Model comparison at a glance

**Scorecard.** Rows = models, columns = slide-level metrics. AUC is threshold-free; all other metrics are at
the validation threshold. PPV = precision (share of slides called inflamed that are inflamed), NPV = share of
slides called non-inflamed that are non-inflamed. The right block is the change from ID to OOD in percentage
points. Grey = not evaluated.

In [ ]:
apply_ieee_style()
W, _ = figsize("double")


def save(fig, name):
    if SAVE_PREFIX:
        for ext in ("png", "pdf"):
            fig.savefig(OUT_DIR / f"{SAVE_PREFIX}_{name}.{ext}", bbox_inches="tight")
        print("Saved:", OUT_DIR / f"{SAVE_PREFIX}_{name}.png")


def panel(ax, letter, title):
    ax.set_title(f"({letter}) {title}", loc="left")


SCORE_METRICS = ["AUC", "Sensitivity", "Specificity", "PPV", "NPV", "Balanced accuracy", "F1"]
SHORT = {"Balanced accuracy": "Bal. acc."}


def score_matrix(dom):
    return np.array([[100 * est[(m, dom)][0][k] if (m, dom) in est else np.nan for k in SCORE_METRICS]
                     for m in models])


mats = {dom: score_matrix(dom) for dom in DOMAINS}
mats["Δ"] = mats["OOD"] - mats["ID"]
dmax = np.nanmax(np.abs(mats["Δ"])) if np.isfinite(mats["Δ"]).any() else 10
dmax = max(10, 10 * np.ceil(dmax / 10))

fig, axes = plt.subplots(1, 3, figsize=(W, 0.36 * len(models) + 1.55), sharey=True)
for ax, key, letter in zip(axes, ["ID", "OOD", "Δ"], "abc"):
    M = mats[key]
    if key == "Δ":
        cmap, vmin, vmax, title = plt.get_cmap("RdBu").copy(), -dmax, dmax, "Change, OOD − ID (pp)"
    else:
        cmap, vmin, vmax, title = plt.get_cmap("Blues").copy(), 40, 100, DOMAIN_LONG[key]
    cmap.set_bad("0.92")
    im = ax.imshow(np.ma.masked_invalid(M), cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            if np.isnan(v):
                ax.text(j, i, "–", ha="center", va="center", fontsize=7, color="0.5")
                continue
            strong = (v > 80) if key != "Δ" else abs(v) > 0.55 * dmax
            ax.text(j, i, f"{v:+.0f}" if key == "Δ" else f"{v:.0f}", ha="center", va="center", fontsize=7,
                    color="white" if strong else "0.1")
    ax.set_xticks(range(len(SCORE_METRICS)), [SHORT.get(k, k) for k in SCORE_METRICS], rotation=40, ha="right")
    ax.set_yticks(range(len(models)), models)
    ax.tick_params(length=0)
    ax.grid(False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks(np.arange(-0.5, len(SCORE_METRICS)), minor=True)
    ax.set_yticks(np.arange(-0.5, len(models)), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5, linestyle="-", alpha=1)
    ax.tick_params(which="minor", length=0)
    panel(ax, letter, title)
    cb = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.06, pad=0.3, aspect=25)
    cb.ax.tick_params(labelsize=6.5)
    cb.outline.set_visible(False)
    cb.set_label("change (pp)" if key == "Δ" else "score (%)", fontsize=7)
fig.text(0.01, 0.0, "AUC is threshold-free; all other metrics at the validation threshold.  – = not evaluated.  "
         "Red = loss, blue = gain under the domain shift.", fontsize=6.5, color="0.35", ha="left", va="bottom")
fig.tight_layout(rect=(0, 0.04, 1, 1))
save(fig, "dg_scorecard")
plt.show()

**Classification outcomes at the validation threshold.** Left: inflamed slides, split into detected (TP) and
missed (FN); the TP share is the sensitivity. Right: non-inflamed slides, split into correctly cleared (TN) and
false alarms (FP); the TN share is the specificity. Numbers are slide counts.

In [ ]:
order = models[::-1]
BAR, GAP = 0.34, 0.2
fig, axes = plt.subplots(1, 2, figsize=(W, 0.62 * len(models) + 1.1), sharey=True)
yticks, yticklabels = [], []
for ax, (good, bad, cls, title, letter) in zip(axes, [("tp", "fn", 1, "Inflamed slides", "a"),
                                                      ("tn", "fp", 0, "Non-inflamed slides", "b")]):
    for yi, model in enumerate(order):
        c = colors[model]
        for dom, dy in (("ID", GAP), ("OOD", -GAP)):
            y = yi + dy
            if ax is axes[0]:
                yticks.append(y)
                yticklabels.append(dom)
            if (model, dom) not in results:
                ax.text(1, y, "not evaluated", va="center", fontsize=6.5, color="0.5")
                continue
            r = results[(model, dom)]
            yv, pv = cohorts[dom]["label"].to_numpy(), r["prob"]
            ok = ~np.isnan(pv)
            cm = confusion(yv[ok], pv[ok], r["at_val"]["threshold"])
            n = cm[good] + cm[bad]
            share = 100 * cm[good] / n
            ax.barh(y, share, height=BAR, color=c, edgecolor="white", linewidth=0.6)
            ax.barh(y, 100 - share, left=share, height=BAR, color=c, alpha=0.18, hatch="////",
                    edgecolor=c, linewidth=0)
            if share > 18:
                ax.text(share - 1.5, y, f"{good.upper()} {cm[good]}", ha="right", va="center", fontsize=6.5,
                        color="white")
            ax.text(101.5, y, f"{share:.0f} %", ha="left", va="center", fontsize=6.5, color="0.1", fontweight="bold")
            ax.text(120, y, f"{bad.upper()} {cm[bad]}", ha="left", va="center", fontsize=6.5,
                    color="0.1" if cm[bad] else "0.55")
    ax.set_xlim(0, 134)
    ax.text(101.5, 1.0, "rate", transform=ax.get_xaxis_transform(), ha="left", va="bottom", fontsize=6.5, color="0.4")
    ax.text(120, 1.0, "errors", transform=ax.get_xaxis_transform(), ha="left", va="bottom", fontsize=6.5,
            color="0.4")
    ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xlabel("% of slides in class")
    ax.grid(axis="y", visible=False)
    ax.set_title(f"({letter}) {title} ({'sensitivity' if cls else 'specificity'})", loc="left", pad=14)
axes[0].set_yticks(yticks, yticklabels, fontsize=7)
axes[0].set_ylim(-0.6, len(order) - 0.4)
for yi, model in enumerate(order):
    axes[0].annotate(model, xy=(0, yi), xycoords=("axes fraction", "data"), xytext=(-24, 0),
                     textcoords="offset points", ha="right", va="center", fontsize=8, color=colors[model],
                     fontweight="bold")
    for ax in axes:
        if yi:
            ax.axhline(yi - 0.5, color="0.85", linewidth=0.6)
from matplotlib.patches import Patch
fig.legend(handles=[Patch(facecolor="0.35", label="correctly classified (TP / TN)"),
                    Patch(facecolor="0.35", alpha=0.18, hatch="////", edgecolor="0.35", linewidth=0,
                          label="misclassified (FN / FP), count under \"errors\"")],
           loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.06, 1, 1))
save(fig, "dg_outcomes")
plt.show()

In [ ]:
# Confusion matrices: rows = domain, columns = model. Colour = row-normalised rate (per true class).
fig, axes = plt.subplots(len(DOMAINS), len(models), figsize=(W, 1.75 * len(DOMAINS) + 0.5), squeeze=False)
for i, dom in enumerate(DOMAINS):
    for j, model in enumerate(models):
        ax = axes[i, j]
        ax.grid(False)
        ax.set_xticks([0, 1], ["inflamed", "non-infl."] if i == len(DOMAINS) - 1 else ["", ""], fontsize=7)
        ax.set_yticks([0, 1], ["inflamed", "non-infl."] if j == 0 else ["", ""], fontsize=7)
        ax.tick_params(length=0)
        if i == 0:
            ax.set_title(model, fontsize=8, color=colors[model], fontweight="bold")
        if j == 0:
            ax.set_ylabel(f"{DOMAIN_LONG[dom]}\ntrue class", fontsize=7.5)
        if i == len(DOMAINS) - 1:
            ax.set_xlabel("predicted", fontsize=7.5)
        if (model, dom) not in results:
            ax.imshow(np.full((2, 2), np.nan), cmap="Blues", vmin=0, vmax=1)
            ax.text(0.5, 0.5, "not evaluated", ha="center", va="center", fontsize=7, color="0.5")
            continue
        r = results[(model, dom)]
        yv, pv = cohorts[dom]["label"].to_numpy(), r["prob"]
        ok = ~np.isnan(pv)
        cm = confusion(yv[ok], pv[ok], r["at_val"]["threshold"])
        counts = np.array([[cm["tp"], cm["fn"]], [cm["fp"], cm["tn"]]])
        rates = counts / counts.sum(axis=1, keepdims=True)
        ax.imshow(rates, cmap="Blues", vmin=0, vmax=1)
        for (a, b), name in zip([(0, 0), (0, 1), (1, 0), (1, 1)], ["TP", "FN", "FP", "TN"]):
            ax.text(b, a, f"{name} {counts[a, b]}\n{100 * rates[a, b]:.0f} %", ha="center", va="center",
                    fontsize=7, color="white" if rates[a, b] > 0.6 else "0.1")
fig.tight_layout(rect=(0, 0, 0.92, 1))
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
cax = fig.add_axes([0.94, 0.18, 0.012, 0.64])
cb = fig.colorbar(ScalarMappable(norm=Normalize(0, 100), cmap="Blues"), cax=cax)
cb.set_label("% of the true class (row)", fontsize=7)
cb.ax.tick_params(labelsize=6.5)
cb.outline.set_visible(False)
save(fig, "dg_confusion")
plt.show()

## 4 · AUC: in-domain vs. out-of-domain

In [ ]:
def domain_legend(include_val=True):
    h = [Line2D([], [], marker="D", markerfacecolor="white", markeredgecolor="0.3", linestyle="",
                label=DOMAIN_LONG["VAL"])] if include_val else []
    return h + [Line2D([], [], marker="o", markerfacecolor="white", markeredgecolor="0.3", linestyle="",
                       label=DOMAIN_LONG["ID"]),
                Line2D([], [], marker="o", color="0.3", linestyle="", label=DOMAIN_LONG["OOD"])]


fig, (ax_a, ax_b) = plt.subplots(1, 2, figsize=(W, 3.2), gridspec_kw={"width_ratios": [1.45, 1]})

# (a) AUC along the domain axis: validation -> ID test -> OOD test
x_pos = {"VAL": 0, "ID": 1, "OOD": 2}
dodge = dict(zip(models, np.linspace(-0.16, 0.16, len(models)) if len(models) > 1 else [0.0]))
for model in models:
    c = colors[model]
    pts = []
    if val_auc[model] is not None:
        pts.append(("VAL", 100 * val_auc[model]))
    for dom in DOMAINS:
        if (model, dom) in est:
            pts.append((dom, 100 * est[(model, dom)][0]["AUC"]))
    for (d0, y0), (d1, y1) in zip(pts, pts[1:]):
        gap = x_pos[d1] - x_pos[d0] > 1
        ax_a.plot([x_pos[d0] + dodge[model], x_pos[d1] + dodge[model]], [y0, y1], color=c, linewidth=1.3,
                  linestyle=":" if gap else model_lines[model], alpha=0.9)
    for d, yv in pts:
        x = x_pos[d] + dodge[model]
        if d == "VAL":
            ax_a.plot(x, yv, model_markers[model], markerfacecolor="white", markeredgecolor=c,
                      markeredgewidth=1.1, markersize=5, zorder=4)
            continue
        lo, hi = 100 * ci(est[(model, d)][1]["AUC"])
        ax_a.plot([x, x], [lo, hi], color=c, linewidth=1.2, alpha=0.75)
        ax_a.plot(x, yv, model_markers[model], markerfacecolor="white" if d == "ID" else c, markeredgecolor=c,
                  markeredgewidth=1.3, markersize=6, zorder=4)
ax_a.set_xticks([0, 1, 2], [DOMAIN_LONG[k].replace(" (", "\n(") for k in ("VAL", "ID", "OOD")])
ax_a.set_xlim(-0.4, 2.4)
ax_a.set_ylabel("Slide-level AUC (%)")
ax_a.grid(axis="x", visible=False)
ax_a.axvspan(1.5, 2.4, color="0.94", zorder=0)   # shaded = domain shift
panel(ax_a, "a", "AUC from validation to OOD test")
fig.legend(handles=[Line2D([], [], color=colors[m], marker=model_markers[m], linestyle=model_lines[m],
                          linewidth=1.3, markersize=5, label=m) for m in models]
           + [Line2D([], [], marker="o", markerfacecolor="white", markeredgecolor="0.3", linestyle="",
                     label="hollow: validation / ID"),
              Line2D([], [], marker="o", color="0.3", linestyle="", label="filled: OOD"),
              Line2D([], [], color="0.4", marker="|", markersize=10, markeredgewidth=1.2, linestyle="",
                     label="vertical / horizontal bar: 95 % CI"),
              __import__("matplotlib.patches", fromlist=["Patch"]).Patch(facecolor="0.94", edgecolor="0.8",
                                                                          label="(a) shaded: OOD domain"),
              Line2D([], [], color="0.3", linestyle=":", label="(b) no change (ΔAUC = 0)")],
           loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))

# (b) ΔAUC (OOD − ID) per model, sorted
deltas = {m: delta_auc(m) for m in models}
order_b = sorted(models, key=lambda m: deltas[m][0] if deltas[m] else -np.inf)
for yi, model in enumerate(order_b):
    if deltas[model] is None:
        ax_b.text(0.02, yi, "ID test not evaluated", transform=ax_b.get_yaxis_transform(), va="center",
                  fontsize=6.5, color="0.45")
        continue
    d, (lo, hi) = deltas[model]
    ax_b.plot([100 * lo, 100 * hi], [yi, yi], color=colors[model], linewidth=2.0, solid_capstyle="round")
    ax_b.plot(100 * d, yi, model_markers[model], color=colors[model], markersize=6.5, markeredgecolor="white",
              zorder=4)
    ax_b.annotate(f"{100 * d:+.1f}", (100 * d, yi), xytext=(0, 6), textcoords="offset points", ha="center",
                  va="bottom", fontsize=6.5, color="0.2")
ax_b.axvline(0, color="0.3", linewidth=0.9, linestyle=":")
ax_b.set_yticks(range(len(order_b)), order_b)
ax_b.set_ylim(-0.6, len(order_b) - 0.4)
ax_b.set_xlabel("ΔAUC, OOD − ID (pp, 95 % CI)")
ax_b.grid(axis="y", visible=False)
panel(ax_b, "b", "Loss under domain shift")
fig.tight_layout(rect=(0, 0.19, 1, 1))
save(fig, "dg_overview")
plt.show()

**Encoder effect on robustness.** ΔΔAUC = (OOD − ID) of an encoder minus (OOD − ID) of the reference
(ResNet-50). Positive = the encoder loses less AUC under the domain shift than ResNet-50. Bootstrap replicates
are paired across encoders within each domain (same resampled patients), independent across domains.

In [ ]:
rows = []
for model in models:
    if model == REFERENCE or not all((m, d) in est for m in (model, REFERENCE) for d in ("ID", "OOD")):
        continue
    reps = ((est[(model, "OOD")][1]["AUC"] - est[(model, "ID")][1]["AUC"])
            - (est[(REFERENCE, "OOD")][1]["AUC"] - est[(REFERENCE, "ID")][1]["AUC"])).to_numpy()
    point = deltas[model][0] - deltas[REFERENCE][0]
    lo, hi = ci(reps)
    reps = reps[~np.isnan(reps)]
    p = min(1.0, 2 * min((reps <= 0).mean(), (reps >= 0).mean()))
    rows.append({"Encoder": model, "ΔAUC encoder (pp)": f"{100 * deltas[model][0]:+.1f}",
                 f"ΔAUC {REFERENCE} (pp)": f"{100 * deltas[REFERENCE][0]:+.1f}",
                 "ΔΔAUC (pp)": f"{100 * point:+.1f}", "95 % CI": f"[{100 * lo:+.1f}, {100 * hi:+.1f}]",
                 "p (bootstrap)": f"{p:.3f}" if p >= 1 / N_BOOT else f"< {1 / N_BOOT:g}"})
robustness = pd.DataFrame(rows).set_index("Encoder") if rows else pd.DataFrame()
print("ID and OOD evaluations of both encoders are needed for ΔΔAUC." if robustness.empty else "")
robustness

**ID vs. OOD AUC.** Each model is one point; the diagonal is "no change under domain shift".
The vertical distance to the diagonal is ΔAUC. Bars are 95 % CIs in both domains.

In [ ]:
both = [m for m in models if all((m, d) in est for d in ("ID", "OOD"))]
fig, ax = plt.subplots(figsize=(W * 0.55, W * 0.5))
lo_all = min(100 * ci(est[(m, "OOD")][1]["AUC"])[0] for m in both) if both else 50
lim = (max(40, np.floor(lo_all / 5) * 5 - 2), 101)
ax.plot(lim, lim, color="0.5", linestyle=":", linewidth=0.9)
ax.text(lim[0] + 3, lim[0] + 4, "no change (OOD = ID)", rotation=45, rotation_mode="anchor", ha="left",
        va="bottom", fontsize=6.5, color="0.45", transform_rotates_text=True)
for model in both:
    (pi, ri), (po, ro) = est[(model, "ID")], est[(model, "OOD")]
    x, y = 100 * pi["AUC"], 100 * po["AUC"]
    (xl, xh), (yl, yh) = 100 * ci(ri["AUC"]), 100 * ci(ro["AUC"])
    ax.plot([xl, xh], [y, y], color=colors[model], linewidth=1.1, alpha=0.7)
    ax.plot([x, x], [yl, yh], color=colors[model], linewidth=1.1, alpha=0.7)
    ax.plot(x, y, model_markers[model], color=colors[model], markersize=7, markeredgecolor="white",
            markeredgewidth=1, zorder=4,
            label=f"{model} (Δ {y - x:+.1f})")
ax.set_xlim(lim)
ax.set_ylim(lim)
ax.set_aspect("equal")
ax.set_xlabel(f"AUC, {DOMAIN_LONG['ID']} (%)")
ax.set_ylabel(f"AUC, {DOMAIN_LONG['OOD']} (%)")
if ax.get_legend_handles_labels()[0]:
    h, l = ax.get_legend_handles_labels()
    h.append(Line2D([], [], color="0.4", marker="+", markersize=9, markeredgewidth=1.0, linestyle="",
                    label="bars: 95 % CI (ID horizontal, OOD vertical)"))
    ax.legend(handles=h, loc="upper left", fontsize=6.5)
missing = [m for m in models if m not in both]
if missing:
    print("Not shown (one domain missing):", ", ".join(missing))
fig.tight_layout()
save(fig, "dg_id_vs_ood")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(DOMAINS), figsize=(W, 3.0), sharey=True)
for ax, dom, letter in zip(axes, DOMAINS, "ab"):
    ax.plot([0, 1], [0, 1], color="0.6", linestyle=":", linewidth=0.8)
    y = cohorts[dom]["label"].to_numpy()
    for model in models:
        if (model, dom) not in results:
            continue
        p = results[(model, dom)]["prob"]
        ok = ~np.isnan(p)
        fpr, tpr, _ = roc_curve(y[ok], p[ok])
        point = est[(model, dom)][0]
        ax.plot(fpr, tpr, color=colors[model], linewidth=1.5, drawstyle="steps-post", linestyle=model_lines[model],
                label=f"{model} ({100 * point['AUC']:.1f})")
        ax.plot(1 - point["Specificity"], point["Sensitivity"], model_markers[model], color=colors[model], markersize=5.5,
                markeredgecolor="white", markeredgewidth=0.8, zorder=4)
    panel(ax, letter, DOMAIN_LONG[dom])
    ax.set_xlabel("1 − specificity")
    ax.set_aspect("equal")
    if ax.get_legend_handles_labels()[0]:
        h, l = ax.get_legend_handles_labels()
        h += [Line2D([], [], color="0.3", marker="o", linestyle="", markersize=5,
                     label="operating point (val. threshold)"),
              Line2D([], [], color="0.6", linestyle=":", label="chance level")]
        ax.legend(handles=h, loc="lower right", fontsize=6, title="AUC (%)", title_fontsize=6.5)
axes[0].set_ylabel("Sensitivity")
fig.tight_layout()
save(fig, "dg_roc")
plt.show()
print("Dots: operating point at the validation threshold.")

## 5 · Scanner shift vs. stain shift

Training used scanner-1 H&E only. **ID·H&E → OOD·H&E** changes only the scanner;
**OOD·H&E → OOD·MG / OOD·PAS** additionally changes the stain. ID·MG and ID·PAS contain no
non-inflamed slides, so their AUC is undefined and they are omitted.

In [ ]:
strata_est = {}
for model in models:
    for dom, stain in STRATA:
        if (model, dom) in results:
            r = results[(model, dom)]
            subset = (cohorts[dom]["stain"] == stain).to_numpy()
            strata_est[(model, dom, stain)] = estimate(dom, r["prob"], r["at_val"]["threshold"], subset)

labels = [f"{dom}\n{'H&E' if s == 'HE' else s}" for dom, s in STRATA]
fig, ax = plt.subplots(figsize=(W, 2.9))
dodge_s = np.linspace(-0.2, 0.2, len(models)) if len(models) > 1 else [0.0]
for k, model in enumerate(models):
    xs, ys = [], []
    for j, (dom, stain) in enumerate(STRATA):
        if (model, dom, stain) not in strata_est:
            continue
        point, reps = strata_est[(model, dom, stain)]
        lo, hi = 100 * ci(reps["AUC"])
        x = j + dodge_s[k]
        ax.plot([x, x], [lo, hi], color=colors[model], linewidth=1.2, alpha=0.7)
        ax.plot(x, 100 * point["AUC"], model_markers[model], markersize=5.5, markeredgecolor=colors[model],
                markeredgewidth=1.2,
                markerfacecolor="white" if dom == "ID" else colors[model], zorder=4)
        xs.append(x); ys.append(100 * point["AUC"])
    ax.plot(xs, ys, linestyle=model_lines[model], color=colors[model], linewidth=1.0, alpha=0.8)
    ax.plot([], [], color=colors[model], linestyle=model_lines[model], marker=model_markers[model],
            markersize=5, label=model)
ax.set_xticks(range(len(STRATA)), labels)
ax.set_ylabel("Slide-level AUC (%)")
ax.grid(axis="x", visible=False)
ax.axvline(0.5, color="0.4", linestyle=":", linewidth=0.9)
ax.axvline(1.5, color="0.75", linestyle=":", linewidth=0.8)
for x, text in ((0, "scanner 1"), (1, "scanner 2 · same stain"), (2.5, "scanner 2 · unseen stains")):
    ax.annotate(text, xy=(x, 1.0), xycoords=("data", "axes fraction"), ha="center", va="bottom",
                fontsize=7, color="0.3")
h, l = ax.get_legend_handles_labels()
h += [Line2D([], [], marker="o", markerfacecolor="white", markeredgecolor="0.3", linestyle="", label="hollow: ID"),
      Line2D([], [], marker="o", color="0.3", linestyle="", label="filled: OOD"),
      Line2D([], [], color="0.4", marker="|", markersize=10, markeredgewidth=1.2, linestyle="", label="95 % CI"),
      Line2D([], [], color="0.4", linestyle=":", label="scanner boundary"),
      Line2D([], [], color="0.75", linestyle=":", label="seen / unseen stain boundary")]
ax.legend(handles=h, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=4, frameon=False)
ax.set_title("Stain-stratified AUC (95 % CI)", pad=14)
fig.tight_layout()
save(fig, "dg_stain")
plt.show()

stain_table = pd.DataFrame({
    model: {lab.replace("\n", " · "): (f"{100 * strata_est[(model, d, s)][0]['AUC']:.1f} "
                                       f"[{100 * ci(strata_est[(model, d, s)][1]['AUC'])[0]:.1f}, "
                                       f"{100 * ci(strata_est[(model, d, s)][1]['AUC'])[1]:.1f}]")
                 if (model, d, s) in strata_est else "–"
            for lab, (d, s) in zip(labels, STRATA)}
    for model in models}).T
stain_table.columns.name = "AUC % [95 % CI]"
stain_table

## 6 · Threshold transfer

AUC measures ranking only. Under domain shift the scores can move relative to the threshold fixed on
scanner-1 validation data. Panels (a, b) show sensitivity and specificity at that threshold. Panel (c) compares
balanced accuracy at the validation threshold (what is deployed) with the **oracle threshold** chosen on the
evaluated set itself (diagnostic only, optimistic): the gap is the part of the error caused by the transferred
threshold, the remaining distance to 100 % is the limited separability (ranking).

In [ ]:
order = models[::-1]
OFF = 0.17
fig, axes = plt.subplots(1, 3, figsize=(W, 0.55 * len(models) + 1.4), sharey=True)
for ax, metric, letter in zip(axes[:2], ["Sensitivity", "Specificity"], "ab"):
    for yi, model in enumerate(order):
        c, pts = colors[model], {}
        for dom, dy, face in (("ID", OFF, "white"), ("OOD", -OFF, c)):
            if (model, dom) not in est:
                continue
            point, reps = est[(model, dom)]
            lo, hi = 100 * ci(reps[metric])
            ax.plot([lo, hi], [yi + dy] * 2, color=c, linewidth=1.2, alpha=0.75)
            ax.plot(100 * point[metric], yi + dy, model_markers[model], markerfacecolor=face, markeredgecolor=c,
                    markeredgewidth=1.3, markersize=5.5, zorder=4)
            pts[dom] = 100 * point[metric]
        if len(pts) == 2:
            ax.annotate("", xy=(pts["OOD"], yi - OFF + 0.06), xytext=(pts["ID"], yi + OFF - 0.06),
                        arrowprops=dict(arrowstyle="-|>", color="0.45", lw=0.7, shrinkA=3, shrinkB=3))
    panel(ax, letter, metric)
    ax.set_xlabel("%")
    ax.set_xlim(-3, 103)
    ax.grid(axis="y", visible=False)

ax = axes[2]
oracle = {}
for yi, model in enumerate(order):
    c = colors[model]
    for dom, dy, face in (("ID", OFF, "white"), ("OOD", -OFF, c)):
        if (model, dom) not in results:
            continue
        y = cohorts[dom]["label"].to_numpy()
        ba_orc, t_orc = oracle_balanced_accuracy(y, results[(model, dom)]["prob"])
        ba_val = est[(model, dom)][0]["Balanced accuracy"]
        oracle[(model, dom)] = (ba_orc, t_orc)
        ax.plot([100 * ba_val, 100 * ba_orc], [yi + dy] * 2, color=c, linewidth=2.2, alpha=0.35,
                solid_capstyle="butt")
        ax.plot(100 * ba_orc, yi + dy, marker="|", color=c, markersize=9, markeredgewidth=1.6, zorder=4)
        ax.plot(100 * ba_val, yi + dy, model_markers[model], markerfacecolor=face, markeredgecolor=c, markeredgewidth=1.3,
                markersize=5.5, zorder=5)
        gap = 100 * (ba_orc - ba_val)
        if gap >= 1:
            ax.annotate(f"−{gap:.0f}", (100 * ba_val, yi + dy), xytext=(-5, 0), textcoords="offset points",
                        ha="right", va="center", fontsize=6.5, color="0.3")
panel(ax, "c", "Balanced accuracy")
ax.set_xlabel("%")
ax.set_xlim(40, 102)
ax.grid(axis="y", visible=False)

axes[0].set_yticks(range(len(order)), order)
axes[0].set_ylim(-0.6, len(order) - 0.4)
fig.suptitle("Operating point at the validation threshold (S1 · H&E)", fontsize=9, y=0.99)
from matplotlib.patches import Patch
fig.legend(handles=domain_legend(include_val=False) + [
    Line2D([], [], color="0.4", linewidth=1.2, label="(a, b) horizontal bar: 95 % CI"),
    Line2D([], [], color="0.45", linewidth=0.7, marker=">", markersize=4, label="(a, b) arrow: shift ID → OOD"),
    Line2D([], [], marker="|", color="0.3", linestyle="", markersize=9, markeredgewidth=1.6,
           label="(c) oracle threshold (diagnostic)"),
    Patch(facecolor="0.6", alpha=0.35, label="(c) bar and number: loss caused by the val. threshold (pp)")],
    loc="lower center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.17, 1, 1))
save(fig, "dg_threshold")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(models), figsize=(W, 2.7), sharey=True, squeeze=False)
jit = np.random.default_rng(SEED)
for ax, model, letter in zip(axes[0], models, "abcdefgh"):
    c = colors[model]
    thr = next(results[(model, d)]["at_val"]["threshold"] for d in DOMAINS if (model, d) in results)
    for j, dom in enumerate(DOMAINS):
        if (model, dom) not in results:
            ax.text(j, 0.5, "not\nevaluated", ha="center", va="center", fontsize=7, color="0.5")
            continue
        p, y = results[(model, dom)]["prob"], cohorts[dom]["label"].to_numpy()
        for cls, dx, face in ((0, -0.17, "white"), (1, 0.17, c)):
            m = (y == cls) & ~np.isnan(p)
            ax.scatter(j + dx + jit.uniform(-0.1, 0.1, m.sum()), p[m], s=10, facecolor=face, edgecolor=c,
                       linewidth=0.8, zorder=3, marker=model_markers[model])
            ax.plot([j + dx - 0.14, j + dx + 0.14], [np.median(p[m])] * 2, color="0.15", linewidth=1.4, zorder=4)
    ax.axhline(thr, color="0.25", linestyle="--", linewidth=0.9)
    ax.set_xticks(range(len(DOMAINS)), list(DOMAINS))
    ax.set_xlim(-0.6, len(DOMAINS) - 0.4)
    ax.set_title(f"({letter}) {model}", loc="left", fontsize=7)
    ax.grid(axis="x", visible=False)
axes[0, 0].set_ylabel("Slide score")
axes[0, 0].set_ylim(-0.03, 1.03)
fig.legend(handles=[
    Line2D([], [], marker="o", markerfacecolor="white", markeredgecolor="0.3", linestyle="", label="non-inflamed"),
    Line2D([], [], marker="o", color="0.3", linestyle="", label="inflamed"),
    Line2D([], [], color="0.15", linewidth=1.4, label="median"),
    Line2D([], [], color="0.25", linestyle="--", label="validation threshold")],
    loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=4, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1))
save(fig, "dg_scores")
plt.show()

## 7 · Results tables

In [ ]:
def fmt(point, reps, key):
    lo, hi = ci(reps[key])
    return f"{100 * point[key]:.1f} [{100 * lo:.1f}, {100 * hi:.1f}]"


rows = []
for model in models:
    for dom in DOMAINS:
        if (model, dom) not in est:
            continue
        point, reps = est[(model, dom)]
        known = ~np.isnan(results[(model, dom)]["prob"])
        row = {"Model": model, "Domain": dom, "Slides": int(known.sum()),
               "Patients": int(cohorts[dom].loc[known, "patient_id"].nunique())}
        row.update({k: fmt(point, reps, k) for k in SCORE_METRICS})
        row["Threshold"] = f"{results[(model, dom)]['at_val']['threshold']:.3f}"
        row["Oracle bal. acc."] = f"{100 * oracle[(model, dom)][0]:.1f}"
        rows.append(row)
    if deltas[model] is not None:
        d, (lo, hi) = deltas[model]
        rows.append({"Model": model, "Domain": "Δ (OOD − ID)", "AUC": f"{100 * d:+.1f} [{100 * lo:+.1f}, {100 * hi:+.1f}]"})

main_table = pd.DataFrame(rows)
for col in ("Slides", "Patients"):
    main_table[col] = main_table[col].astype("Int64").astype(str).replace("<NA>", "")
main_table = main_table.set_index(["Model", "Domain"]).fillna("")
main_table.columns.name = "% [95 % CI]"
main_table

In [ ]:
# Pairwise comparison with the reference model inside each domain (paired patient-level bootstrap on the
# slides predicted by both models), Holm-corrected over all comparisons.
def holm(pvals):
    p = np.asarray(pvals, float)
    adj, running = np.empty_like(p), 0.0
    for rank, i in enumerate(np.argsort(p)):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


def fmt_p(p):
    # A bootstrap with B resamples cannot resolve p below 1/B.
    return f"< {1 / N_BOOT:.4f}".rstrip("0") if p < 1 / N_BOOT else f"{p:.3f}"


rows = []
for dom in DOMAINS:
    if (REFERENCE, dom) not in results:
        continue
    y = cohorts[dom]["label"].to_numpy()
    p_ref = results[(REFERENCE, dom)]["prob"]
    for model in models:
        if model == REFERENCE or (model, dom) not in results:
            continue
        p_mod = results[(model, dom)]["prob"]
        common = ~np.isnan(p_ref) & ~np.isnan(p_mod)
        diffs = np.asarray([roc_auc_score(y[i], p_mod[i]) - roc_auc_score(y[i], p_ref[i])
                            for i in (idx[common[idx]] for idx in boot[dom]) if 0 < y[i].sum() < len(i)])
        delta = roc_auc_score(y[common], p_mod[common]) - roc_auc_score(y[common], p_ref[common])
        rows.append({"Domain": dom, "Model": model, "Common slides": int(common.sum()),
                     "ΔAUC vs. reference (pp)": f"{100 * delta:+.1f}",
                     "95 % CI": f"[{100 * np.percentile(diffs, 2.5):+.1f}, {100 * np.percentile(diffs, 97.5):+.1f}]",
                     "p": min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean()))})

pairwise = pd.DataFrame(rows)
if len(pairwise):
    pairwise["p (Holm)"] = holm(pairwise["p"])
    pairwise["p"] = pairwise["p"].map(fmt_p)
    pairwise["p (Holm)"] = pairwise["p (Holm)"].map(fmt_p)
    pairwise = pairwise.set_index(["Domain", "Model"])
print(f"Reference: {REFERENCE}")
pairwise

## 8 · Key results (generated)

In [ ]:
lines = []
for model in models:
    parts = []
    for dom in DOMAINS:
        if (model, dom) in est:
            lo, hi = 100 * ci(est[(model, dom)][1]["AUC"])
            parts.append(f"{dom} AUC {100 * est[(model, dom)][0]['AUC']:.1f} [{lo:.1f}, {hi:.1f}]")
        else:
            parts.append(f"{dom} not evaluated")
    if deltas[model] is not None:
        d, (lo, hi) = deltas[model]
        sig = "CI excludes 0" if hi < 0 or lo > 0 else "CI includes 0"
        parts.append(f"ΔAUC {100 * d:+.1f} [{100 * lo:+.1f}, {100 * hi:+.1f}] ({sig})")
    if (model, "OOD") in est:
        spec = est[(model, "OOD")][0]["Specificity"]
        gap = oracle[(model, "OOD")][0] - est[(model, "OOD")][0]["Balanced accuracy"]
        parts.append(f"OOD specificity at val. threshold {100 * spec:.1f}, threshold cost {100 * gap:.1f} pp bal. acc.")
    lines.append(f"- {model}: " + "; ".join(parts))
print("\n".join(lines))
if len(pairwise):
    print(f"\nPairwise vs. {REFERENCE} (Holm-corrected):")
    for (dom, model), r in pairwise.iterrows():
        p_txt = r["p (Holm)"] if r["p (Holm)"].startswith("<") else f"= {r['p (Holm)']}"
        print(f"- {dom}: {model} {r['ΔAUC vs. reference (pp)']} pp {r['95 % CI']}, p {p_txt}")

In [ ]:
if SAVE_PREFIX:
    for name, table in (("cohort", cohort_table), ("main", main_table), ("stain_auc", stain_table),
                        ("pairwise", pairwise)):
        table.to_csv(OUT_DIR / f"{SAVE_PREFIX}_table_{name}.csv")
        table.to_latex(OUT_DIR / f"{SAVE_PREFIX}_table_{name}.tex", escape=True)
    print("Tables written to", OUT_DIR)